# Calling Your Own Function

In the previous notebook, OpenAI provided the web search tool. Here, **we provide a Python function** that the model can request.

The model can answer directly or ask to use our function. When it asks, our notebook runs the function and sends its result back for a final answer.

In [1]:
import json

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

client = OpenAI()
model = "gpt-6-luna"

## 1. Define the function

This function returns 25 AI engineering skills with hardcoded example percentages. The values are **illustrative lesson data**, not current job-market statistics.

The values stay the same every time you run the function.

In [2]:
def in_demand_ai_skills():
    return [
        {"skill": "python", "percentage": 88.1},
        {"skill": "llm", "percentage": 57.1},
        {"skill": "rag", "percentage": 45.2},
        {"skill": "aws", "percentage": 40.5},
        {"skill": "prompt engineering", "percentage": 35.7},
        {"skill": "langchain", "percentage": 28.6},
        {"skill": "azure", "percentage": 23.8},
        {"skill": "machine learning", "percentage": 19.0},
        {"skill": "generative ai", "percentage": 19.0},
        {"skill": "gcp", "percentage": 19.0},
        {"skill": "ai agents", "percentage": 19.0},
        {"skill": "tensorflow", "percentage": 16.7},
        {"skill": "fine-tuning", "percentage": 16.7},
        {"skill": "pytorch", "percentage": 16.7},
        {"skill": "vector databases", "percentage": 14.3},
        {"skill": "nlp", "percentage": 14.3},
        {"skill": "docker", "percentage": 14.3},
        {"skill": "typescript", "percentage": 14.3},
        {"skill": "langgraph", "percentage": 14.3},
        {"skill": "sql", "percentage": 11.9},
        {"skill": "kubernetes", "percentage": 11.9},
        {"skill": "llamaindex", "percentage": 11.9},
        {"skill": "cicd", "percentage": 11.9},
        {"skill": "mcp", "percentage": 11.9},
        {"skill": "embeddings", "percentage": 11.9},
    ]

In [4]:
in_demand_ai_skills()[:5]

[{'skill': 'python', 'percentage': 88.1},
 {'skill': 'llm', 'percentage': 57.1},
 {'skill': 'rag', 'percentage': 45.2},
 {'skill': 'aws', 'percentage': 40.5},
 {'skill': 'prompt engineering', 'percentage': 35.7}]

## 2. Describe the function to the model

The tool definition tells the model the function's **name**, **purpose**, and **arguments**. This function takes no arguments, so its parameter schema is an empty object. The function's Python code and data stay in our notebook.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "in_demand_ai_skills",
        "description": "Use for questions about demand for AI engineering skills. Return 25 example skills with illustrative percentages, sorted from highest to lowest.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    }
]

## 3. Let the model decide

With `tool_choice="auto"`, the model can answer directly or request our function. The question below is unrelated to AI engineering skills, so the model should answer it without a function call.

Try a question about in-demand AI engineering skills afterward to see the other path.

In [ ]:
question = "How big is the sun?"
conversation = [{"role": "user", "content": question}]

response = client.responses.create(
    model=model,
    input=conversation,
    tools=tools,
    tool_choice="auto",
)

tool_calls = [item for item in response.output if item.type == "function_call"]

if tool_calls:
    for tool_call in tool_calls:
        print(f"Requested function: {tool_call.name}")
        print(f"Arguments: {tool_call.arguments}")
else:
    print("The model answered without calling a function:")
    print(response.output_text)

## 4. Run the function if requested

A function call is only a request from the model. If there is one, **our notebook runs the Python function** and prepares its result. Otherwise, there is nothing to run.

In [ ]:
tool_results = []

for tool_call in tool_calls:
    if tool_call.name != "in_demand_ai_skills":
        raise ValueError(f"Unexpected function: {tool_call.name}")

    skills = in_demand_ai_skills()
    tool_results.append(
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(skills),
        }
    )
    print(f"Python returned {len(skills)} skills.")

## 5. Send any function results back to the model

If the model requested a function, we add its output and our result to the conversation list. The `call_id` links each result to its request. We then send the complete list in a second call.

If the model answered directly, there is no result to send and no second call. Our code manages the history, a pattern we can use with other providers even though their tool-message formats differ.

In [ ]:
if tool_results:
    conversation.extend(response.output)
    conversation.extend(tool_results)

    final_response = client.responses.create(
        model=model,
        input=conversation,
    )
    print(final_response.output_text)
else:
    print("No second request needed.")

## Your turn

Change the question in step 3 to ask which **cloud skills** appear in the example data. Run steps 3–5 again. This time, the model should request the function and use its result. Tool choice is still the model's decision, so it may occasionally choose differently.

The key distinction: **function calling lets the model request your code. Your application executes that code and returns the result.** See the [OpenAI function-calling guide](https://developers.openai.com/api/docs/guides/function-calling).